# Notebook: Enriquecimento de Leads com Base BB
Células prontas para copiar e colar no Databricks

**Conexões:**
- Leads : `main.main.leads_coluna_origem`
- Base BB: `corporativos_pd.db2mci.cliente`

In [ ]:
# CÉLULA 1 — Imports e constantes
import re
import unicodedata
from pyspark import StorageLevel
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

CORRUPT_RECORD_COLUMN = "_corrupt_record"
MIN_VALID_DATE = "1900-01-01"
SENTINEL_DATE = "0001-01-01"
KNOWN_INVALID_CPFS = {"12345678909"}
DATE_STYLE_OPTIONS = {"AUTO_SAFE", "DMY", "MDY"}

FINAL_OUTPUT_COLUMNS = [
    "evento",
    "data_evento",
    "Soma de ano_evento",
    "cliente",
    "cod_sexo",
    "cpf",
    "data_nascimento",
    "faixa_etaria",
    "Soma de idade",
    "local",
    "tipo_evento",
]

CONTEXT_COLUMNS = ["evento", "tipo_evento", "local", "data_evento"]
LEAD_STAGING_COLUMNS = set(CONTEXT_COLUMNS + ["cpf", "data_nascimento"])

HEADER_ALIASES = {
    "dt_nascimento": "data_nascimento",
    "evento_nome": "evento",
}

NULL_TOKENS = ["", "nan", "nat", "none", "null"]

ISSUES_COLUMNS = [
    "__lead_row_id",
    "severity",
    "issue_code",
    "source",
    "field_name",
    "raw_value",
    "normalized_value",
]

In [ ]:
# CÉLULA 2 — Parâmetros fixos (sem widgets)
# Ajuste os valores abaixo conforme necessário antes de executar.
BB_TABLE_NAME = "corporativos_pd.db2mci.cliente" # tabela BB
LEADS_TABLE_NAME = "main.main.leads_coluna_origem" # tabela de leads
OUTPUT_FINAL_TABLE = "" # ex: "main.main.leads_enriquecidos" — deixe vazio para não salvar
OUTPUT_AUDIT_TABLE = "" # ex: "main.main.leads_audit"
OUTPUT_ISSUES_TABLE = "" # ex: "main.main.leads_issues"

LEAD_DATE_STYLE = "AUTO_SAFE" # AUTO_SAFE | DMY | MDY

# Valores padrão usados quando a tabela de leads não possui a coluna correspondente
DEFAULT_CONTEXT = {
    "evento": "",
    "tipo_evento": "",
    "local": "",
    "data_evento": "",
}

if LEAD_DATE_STYLE not in DATE_STYLE_OPTIONS:
    raise ValueError(f"LEAD_DATE_STYLE inválido: {LEAD_DATE_STYLE}. Use AUTO_SAFE, DMY ou MDY")

print("✅ Parâmetros carregados com sucesso.")
print(f" BB_TABLE_NAME : {BB_TABLE_NAME}")
print(f" LEADS_TABLE_NAME : {LEADS_TABLE_NAME}")
print(f" LEAD_DATE_STYLE : {LEAD_DATE_STYLE}")

In [ ]:
# CÉLULA 3 — Funções auxiliares puras
def strip_accents(value):
    normalized = unicodedata.normalize("NFKD", str(value or ""))
    return "".join(char for char in normalized if not unicodedata.combining(char))

def normalize_header(header):
    value = strip_accents(str(header or "").lstrip("\ufeff").strip().lower())
    value = re.sub(r"[^a-z0-9]+", "_", value).strip("_")
    return HEADER_ALIASES.get(value, value)

def safe_col(column_name):
    escaped = str(column_name).replace("`", "``")
    return F.col(f"`{escaped}`")

def calc_cpf_check_digit(numbers, start_weight):
    total = 0
    weight = start_weight
    for number in numbers:
        total += number * weight
        weight -= 1
    remainder = total % 11
    return 0 if remainder < 2 else 11 - remainder

def is_valid_cpf(value):
    digits = re.sub(r"\D+", "", str(value or ""))
    if len(digits) != 11:
        return False
    if digits == digits[0] * 11:
        return False
    if digits in KNOWN_INVALID_CPFS:
        return False
    numbers = [int(c) for c in digits]
    first = calc_cpf_check_digit(numbers[:9], 10)
    second = calc_cpf_check_digit(numbers[:10], 11)
    return first == numbers[9] and second == numbers[10]

validate_cpf_udf = F.udf(is_valid_cpf, T.BooleanType())

def try_timestamp(column_name, pattern):
    escaped = str(column_name).replace("`", "``")
    pattern_escaped = str(pattern).replace("'", "\\'")
    return F.expr(f"try_to_timestamp(`{escaped}`, '{pattern_escaped}')")

def empty_issues_df():
    schema = T.StructType([
        T.StructField("__lead_row_id", T.StringType(), True),
        T.StructField("severity", T.StringType(), False),
        T.StructField("issue_code", T.StringType(), False),
        T.StructField("source", T.StringType(), False),
        T.StructField("field_name", T.StringType(), True),
        T.StructField("raw_value", T.StringType(), True),
        T.StructField("normalized_value", T.StringType(), True),
    ])
    return spark.createDataFrame([], schema=schema)

print("✅ Funções auxiliares definidas.")

In [ ]:
# CÉLULA 4 — Leitura da tabela de leads (Unity Catalog)
import re as _re
import unicodedata as _unicodedata

df_leads_raw = spark.read.table(LEADS_TABLE_NAME)

# Normaliza cabeçalhos e garante unicidade
raw_cols = df_leads_raw.columns
normalized_cols = []
seen = {}
for col in raw_cols:
    base = normalize_header(col)
    if base not in seen:
        normalized_cols.append(base)
        seen[base] = 1
    else:
        new_name = f"{base}_{seen[base]}"
        normalized_cols.append(new_name)
        seen[base] += 1

raw_leads_df = df_leads_raw.toDF(*normalized_cols)

# Verifica duplicatas após normalização
duplicates = sorted({c for c in normalized_cols if normalized_cols.count(c) > 1})
if duplicates:
    raise ValueError(f"Cabeçalhos duplicados após normalização: {', '.join(duplicates)}")

# Verifica coluna obrigatória de CPF
if "cpf" not in raw_leads_df.columns:
    raise ValueError("Coluna obrigatória ausente após normalização: cpf")

# Contagem de corrompidos (não aplicável para tabelas UC — apenas registra 0)
corrupt_rows_count = 0
print(f"✅ Tabela de leads carregada: {LEADS_TABLE_NAME}")
print(f" Colunas normalizadas ({len(raw_leads_df.columns)}): {raw_leads_df.columns}")
print(f" Linhas corrompidas: {corrupt_rows_count}")

In [ ]:
# CÉLULA 5 — Staging dos leads
# --- DEFINIÇÕES ATUALIZADAS ---
LEAD_STAGING_COLUMNS = [
    "nome", "cpf", "data_nascimento", "email", "telefone",
    "origem", "evento", "local", "data_evento", "tipo_evento"
]
CONTEXT_COLUMNS = ["tipo_evento", "origem"]
DEFAULT_CONTEXT = {
    "tipo_evento": "Não Informado",
    "origem": "Não Informada"
}
LEADS_TABLE_NAME_STAGING = "leads_raw"
# --- FIM DAS DEFINIÇÕES ---

source_columns = raw_leads_df.columns
lead_staging_pairs = [
    (col, col)
    for col in source_columns
    if col in LEAD_STAGING_COLUMNS
]

staging_leads_df = raw_leads_df.select(
    *[F.col(src).alias(tgt) for src, tgt in lead_staging_pairs],
    F.lit(LEADS_TABLE_NAME_STAGING).alias("__input_file_name"),
)

missing_context = []
for column in CONTEXT_COLUMNS:
    if column not in staging_leads_df.columns:
        if DEFAULT_CONTEXT.get(column) is not None:
            staging_leads_df = staging_leads_df.withColumn(column, F.lit(DEFAULT_CONTEXT[column]).cast("string"))
        else:
            missing_context.append(column)

if missing_context:
    raise ValueError("Contexto obrigatório ausente e sem default: " + ", ".join(missing_context))

if "data_nascimento" not in staging_leads_df.columns:
    staging_leads_df = staging_leads_df.withColumn("data_nascimento", F.lit(None).cast("string"))

staging_leads_df = (
    staging_leads_df
    .withColumn("__row_uid", F.monotonically_increasing_id())
    .withColumn(
        "__lead_row_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("__input_file_name"), F.lit("")),
                F.col("__row_uid").cast("string"),
            ),
            256,
        ),
    )
    .drop("__row_uid")
    .withColumn("__cpf_raw", F.col("cpf").cast("string"))
    .withColumn("__data_evento_raw", F.col("data_evento").cast("string"))
    .withColumn("__data_nascimento_raw", F.col("data_nascimento").cast("string"))
)

for column in ["evento", "tipo_evento", "local", "data_evento", "data_nascimento", "origem"]:
    if column in staging_leads_df.columns:
        staging_leads_df = staging_leads_df.withColumn(column, F.col(column).cast("string"))
    else:
        staging_leads_df = staging_leads_df.withColumn(column, F.lit(None).cast("string"))

staging_leads_df.createOrReplaceTempView("staging_leads_df_raw")
print("✅ Staging dos leads criada com a coluna 'origem'.")

In [ ]:
# CÉLULA 6 — Padronização e validação de CPF dos leads
cpf_digits = F.regexp_replace(F.coalesce(F.col("__cpf_raw"), F.lit("")), r"[^0-9]", "")
cpf_digits_len = F.length(cpf_digits)
cpf_normalized = F.when(cpf_digits_len.between(1, 11), F.lpad(cpf_digits, 11, "0"))
cpf_repeated = cpf_normalized.rlike(r"^(\d)\1{10}$")
cpf_issue_code = (
    F.when(cpf_digits_len == 0, F.lit("CPF_EMPTY"))
    .when(cpf_digits_len > 11, F.lit("CPF_GT_11"))
    .when(cpf_repeated, F.lit("CPF_REPEATED_DIGITS"))
    .when(cpf_normalized == F.lit("12345678909"), F.lit("CPF_KNOWN_PLACEHOLDER"))
    .when(~validate_cpf_udf(cpf_normalized), F.lit("CPF_CHECK_DIGIT_INVALID"))
)

staging_leads_df = (
    staging_leads_df
    .withColumn("cpf_digits_raw", cpf_digits)
    .withColumn("cpf_digits_len", cpf_digits_len)
    .withColumn("cpf_normalizado_11", cpf_normalized)
    .withColumn("cpf_lpad_applied", cpf_digits_len.between(1, 10))
    .withColumn("cpf_issue_code", cpf_issue_code)
    .withColumn("cpf_valido", F.col("cpf_issue_code").isNull())
    .withColumn(
        "cpf_output",
        F.when(cpf_digits_len.between(1, 11), F.col("cpf_normalizado_11")).otherwise(F.lit(None).cast("string")),
    )
)
print("✅ CPF dos leads padronizado e validado.")

In [ ]:
# CÉLULA 7 — Funções de limpeza e parsing de datas
def clean_text_column(column_name):
    cleaned = F.trim(F.col(column_name))
    return F.when(F.lower(cleaned).isin(*NULL_TOKENS), F.lit(None).cast("string")).otherwise(cleaned)

def add_date_parse_columns(df, raw_column, prefix, date_kind):
    clean_col = f"{prefix}_clean"
    candidate_col = f"{prefix}_candidate"
    status_col = f"{prefix}_status"
    valid_col = f"{prefix}_valid"
    df = df.withColumn(clean_col, clean_text_column(raw_column))
    
    iso_candidate = F.to_date(
        F.coalesce(
            try_timestamp(clean_col, "yyyy-MM-dd HH:mm:ss.SSSSSS"),
            try_timestamp(clean_col, "yyyy-MM-dd HH:mm:ss.SSS"),
            try_timestamp(clean_col, "yyyy-MM-dd HH:mm:ss"),
            try_timestamp(clean_col, "yyyy-MM-dd"),
        )
    )
    
    slash_like = F.col(clean_col).rlike(r"^\d{1,2}/\d{1,2}/\d{4}(?: \d{1,2}:\d{1,2}(?::\d{1,2})?)?$")
    first_token = F.regexp_extract(F.col(clean_col), r"^(\d{1,2})/", 1).cast("int")
    second_token = F.regexp_extract(F.col(clean_col), r"^\d{1,2}/(\d{1,2})/", 1).cast("int")
    ambiguous_slash = slash_like & first_token.between(1, 12) & second_token.between(1, 12)
    
    dmy_candidate = F.to_date(F.coalesce(
        try_timestamp(clean_col, "d/M/yyyy H:m:s"),
        try_timestamp(clean_col, "d/M/yyyy H:m"),
        try_timestamp(clean_col, "d/M/yyyy"),
    ))
    
    mdy_candidate = F.to_date(F.coalesce(
        try_timestamp(clean_col, "M/d/yyyy H:m:s"),
        try_timestamp(clean_col, "M/d/yyyy H:m"),
        try_timestamp(clean_col, "M/d/yyyy"),
    ))
    
    if LEAD_DATE_STYLE == "DMY":
        slash_candidate = F.when(slash_like, dmy_candidate)
        is_ambiguous = F.lit(False)
    elif LEAD_DATE_STYLE == "MDY":
        slash_candidate = F.when(slash_like, mdy_candidate)
        is_ambiguous = F.lit(False)
    else:
        slash_candidate = (
            F.when(slash_like & (first_token > 12), dmy_candidate)
            .when(slash_like & (second_token > 12), mdy_candidate)
        )
        is_ambiguous = ambiguous_slash
        
    candidate = F.coalesce(iso_candidate, slash_candidate)
    min_date = F.to_date(F.lit(MIN_VALID_DATE))
    
    if date_kind == "birth":
        status = (
            F.when(F.col(clean_col).isNull(), F.lit("MISSING"))
            .when(is_ambiguous, F.lit("AMBIGUOUS"))
            .when(candidate.isNull(), F.lit("UNPARSEABLE"))
            .when(candidate < min_date, F.lit("BEFORE_MIN"))
            .when(candidate > F.current_date(), F.lit("FUTURE"))
            .otherwise(F.lit("VALID"))
        )
    else:
        status = (
            F.when(F.col(clean_col).isNull(), F.lit("MISSING"))
            .when(is_ambiguous, F.lit("AMBIGUOUS"))
            .when(candidate.isNull(), F.lit("UNPARSEABLE"))
            .when(candidate < min_date, F.lit("BEFORE_MIN"))
            .otherwise(F.lit("VALID"))
        )
        
    return (
        df.withColumn(candidate_col, candidate)
        .withColumn(status_col, status)
        .withColumn(valid_col, F.when(F.col(status_col) == F.lit("VALID"), F.col(candidate_col)))
    )

print("✅ Funções de parsing de datas definidas.")

In [ ]:
# CÉLULA 8 — Parsing e validação das datas dos leads
staging_leads_df = add_date_parse_columns(
    staging_leads_df, "__data_nascimento_raw", "lead_birthdate", "birth"
)
staging_leads_df = add_date_parse_columns(
    staging_leads_df, "__data_evento_raw", "lead_event_date", "event"
)

staging_metrics = staging_leads_df.agg(
    F.count(F.lit(1)).alias("input_leads_count"),
    F.sum(
        F.when(F.col("lead_event_date_status") == F.lit("VALID"), F.lit(1)).otherwise(F.lit(0))
    ).alias("valid_event_rows"),
).collect()[0]

input_leads_count = int(staging_metrics["input_leads_count"])
valid_event_rows = int(staging_metrics["valid_event_rows"] or 0)

if valid_event_rows == 0:
    raise ValueError("Nenhuma linha possui data_evento válida. Notebook abortado.")

staging_leads_df.createOrReplaceTempView("staging_leads_df")
print(f"✅ Datas parseadas. Total de leads: {input_leads_count} | Com data_evento válida: {valid_event_rows}")

In [ ]:
# CÉLULA 9 — Shortlist de CPFs elegíveis para match
lead_cpfs_matchable = (
    staging_leads_df
    .filter(F.col("cpf_valido"))
    .select(F.col("cpf_normalizado_11").alias("cpf_match"))
    .distinct()
)
lead_cpfs_matchable.createOrReplaceTempView("lead_cpfs_matchable")
print(f"✅ CPFs únicos elegíveis para match: {lead_cpfs_matchable.count()}")

In [ ]:
# CÉLULA 10 — Leitura, filtro técnico e deduplicação da base BB
bb_required_columns = {"cod_tipo", "cod_cpf_cgc", "dta_nasc_csnt", "dta_ulta_atlz", "dta_revisao", "cod"}
bb_available_columns = set(spark.table(BB_TABLE_NAME).columns)
bb_missing_columns = sorted(bb_required_columns - bb_available_columns)

if bb_missing_columns:
    raise ValueError("Base BB sem colunas obrigatórias: " + ", ".join(bb_missing_columns))

bb_base_df = (
    spark.table(BB_TABLE_NAME)
    .select("cod_tipo", "cod_cpf_cgc", "dta_nasc_csnt", "dta_ulta_atlz", "dta_revisao", "cod")
    .where(F.col("cod_tipo") == F.lit(1))
    .where(F.col("cod_cpf_cgc").isNotNull())
    .withColumn("bb_cpf_text", F.col("cod_cpf_cgc").cast("string"))
    .withColumn("bb_cpf_digits", F.regexp_replace(F.col("bb_cpf_text"), r"[^0-9]", ""))
    .withColumn("bb_cpf_len", F.length(F.col("bb_cpf_digits")))
    .where(F.col("bb_cpf_len").between(1, 11))
    .withColumn("bb_cpf_normalizado_11", F.lpad(F.col("bb_cpf_digits"), 11, "0"))
)

bb_pruned_df = (
    bb_base_df.join(
        F.broadcast(lead_cpfs_matchable),
        bb_base_df["bb_cpf_normalizado_11"] == lead_cpfs_matchable["cpf_match"],
        "left_semi",
    )
    .withColumn("bb_cpf_valido", validate_cpf_udf(F.col("bb_cpf_normalizado_11")))
    .where(F.col("bb_cpf_valido"))
)

sentinel_date = F.to_date(F.lit(SENTINEL_DATE))
min_date = F.to_date(F.lit(MIN_VALID_DATE))

bb_pruned_df = (
    bb_pruned_df
    .withColumn(
        "bb_dta_ulta_atlz_ord",
        F.when(F.col("dta_ulta_atlz") == sentinel_date, F.lit(None).cast("date")).otherwise(F.col("dta_ulta_atlz")),
    )
    .withColumn(
        "bb_dta_revisao_ord",
        F.when(F.col("dta_revisao") == sentinel_date, F.lit(None).cast("date")).otherwise(F.col("dta_revisao")),
    )
    .withColumn(
        "bb_dta_nasc_valid",
        F.when(F.col("dta_nasc_csnt") == sentinel_date, F.lit(None).cast("date"))
        .when(F.col("dta_nasc_csnt") < min_date, F.lit(None).cast("date"))
        .when(F.col("dta_nasc_csnt") > F.current_date(), F.lit(None).cast("date"))
        .otherwise(F.col("dta_nasc_csnt")),
    )
    .withColumn(
        "bb_tiebreak_hash",
        F.sha2(
            F.to_json(F.struct("bb_cpf_normalizado_11", "dta_ulta_atlz", "dta_revisao", "cod")),
            256,
        ),
    )
)

bb_rows_after_prune = bb_pruned_df.count()

bb_window = Window.partitionBy("bb_cpf_normalizado_11").orderBy(
    F.col("bb_dta_ulta_atlz_ord").desc_nulls_last(),
    F.col("bb_dta_revisao_ord").desc_nulls_last(),
    F.col("cod").desc_nulls_last(),
    F.col("bb_tiebreak_hash").desc(),
)

bb_dedup_df = (
    bb_pruned_df
    .withColumn("bb_row_num", F.row_number().over(bb_window))
    .where(F.col("bb_row_num") == F.lit(1))
    .drop("bb_row_num")
)

bb_dedup_metrics = bb_dedup_df.agg(
    F.count(F.lit(1)).alias("bb_rows_after_dedupe"),
    F.max("bb_dta_ulta_atlz_ord").alias("bb_max_dta_ulta_atlz"),
).collect()[0]

bb_rows_after_dedupe = int(bb_dedup_metrics["bb_rows_after_dedupe"])
bb_max_dta_ulta_atlz = bb_dedup_metrics["bb_max_dta_ulta_atlz"]
bb_max_dta_ulta_atlz = str(bb_max_dta_ulta_atlz) if bb_max_dta_ulta_atlz is not None else None

bb_dedup_df.createOrReplaceTempView("bb_dedup_df")
print(f"✅ Base BB processada.")
print(f" Linhas após filtro : {bb_rows_after_prune}")
print(f" Linhas após dedupe : {bb_rows_after_dedupe}")
print(f" Máx dta_ulta_atlz : {bb_max_dta_ulta_atlz}")

In [ ]:
# CÉLULA 11 — Join leads ← BB e reconciliação dos campos
joined_df = staging_leads_df.join(
    bb_dedup_df,
    (staging_leads_df["cpf_valido"])
    & (staging_leads_df["cpf_normalizado_11"] == bb_dedup_df["bb_cpf_normalizado_11"]),
    "left",
)

joined_df = (
    joined_df
    .withColumn("cliente", F.col("bb_cpf_normalizado_11").isNotNull())
    .withColumn(
        "match_status",
        F.when(~F.col("cpf_valido"), F.lit("INVALID_CPF"))
        .when(F.col("bb_cpf_normalizado_11").isNotNull(), F.lit("MATCHED"))
        .otherwise(F.lit("NO_MATCH")),
    )
    .withColumn("data_nascimento_final",
        F.coalesce(F.col("bb_dta_nasc_valid"), F.col("lead_birthdate_valid"))
    )
    .withColumn(
        "birthdate_source",
        F.when(F.col("bb_dta_nasc_valid").isNotNull(), F.lit("bb"))
        .when(F.col("lead_birthdate_valid").isNotNull(), F.lit("lead"))
        .otherwise(F.lit("none")),
    )
    .withColumn("data_evento_final", F.col("lead_event_date_valid"))
    .withColumn("cod_sexo_final", F.lit(None).cast("string"))
    .withColumn(
        "required_context_missing",
        (F.length(F.trim(F.coalesce(F.col("evento"), F.lit("")))) == 0)
        | (F.length(F.trim(F.coalesce(F.col("tipo_evento"), F.lit("")))) == 0)
        | (F.length(F.trim(F.coalesce(F.col("local"), F.lit("")))) == 0),
    )
)

print("✅ Join realizado.")

In [ ]:
# CÉLULA 12 — Derivações de negócio e montagem do final_df
joined_df = (
    joined_df
    .withColumn("Soma de ano_evento", F.year(F.col("data_evento_final")).cast("int"))
    .withColumn(
        "Soma de idade",
        F.when(
            F.col("data_evento_final").isNotNull() & F.col("data_nascimento_final").isNotNull(),
            (F.year(F.col("data_evento_final")) - F.year(F.col("data_nascimento_final"))).cast("int"),
        ).otherwise(F.lit(None).cast("int")),
    )
    .withColumn(
        "faixa_etaria",
        F.when(F.col("Soma de idade").isNull(), F.lit("Desconhecido"))
        .when(F.col("Soma de idade") < 18, F.lit("<18"))
        .when(F.col("Soma de idade") <= 40, F.lit("18-40"))
        .otherwise(F.lit("40+")),
    )
)

joined_metrics = joined_df.agg(
    F.count(F.lit(1)).alias("joined_count"),
    F.countDistinct("__lead_row_id").alias("unique_row_ids"),
).collect()[0]

joined_count = int(joined_metrics["joined_count"])
unique_row_ids = int(joined_metrics["unique_row_ids"])

if joined_count != input_leads_count:
    raise ValueError(f"Join alterou cardinalidade: entrada={input_leads_count}, saída={joined_count}")
if unique_row_ids != input_leads_count:
    raise ValueError(f"__lead_row_id perdeu unicidade: entrada={input_leads_count}, únicos={unique_row_ids}")

joined_df.createOrReplaceTempView("joined_df")

final_df = joined_df.select(
    F.col("evento").alias("evento"),
    F.col("data_evento_final").alias("data_evento"),
    F.col("Soma de ano_evento"),
    F.col("cliente"),
    F.col("cod_sexo_final").alias("cod_sexo"),
    F.col("cpf_output").alias("cpf"),
    F.col("data_nascimento_final").alias("data_nascimento"),
    F.col("faixa_etaria"),
    F.col("Soma de idade"),
    F.col("local"),
    F.col("tipo_evento"),
)

final_df.createOrReplaceTempView("final_df")
print(f"✅ final_df montado. Linhas: {joined_count}")

In [ ]:
# CÉLULA 13 — Montagem do issues_df
issues_df = empty_issues_df()

cpf_issues_df = joined_df.filter(F.col("cpf_issue_code").isNotNull()).select(
    F.col("__lead_row_id"),
    F.lit("error").alias("severity"),
    F.col("cpf_issue_code").alias("issue_code"),
    F.lit("leads").alias("source"),
    F.lit("cpf").alias("field_name"),
    F.col("__cpf_raw").cast("string").alias("raw_value"),
    F.col("cpf_output").cast("string").alias("normalized_value"),
)

birth_ambiguous_df = joined_df.filter(F.col("lead_birthdate_status") == F.lit("AMBIGUOUS")).select(
    F.col("__lead_row_id"),
    F.lit("error").alias("severity"),
    F.lit("DATE_AMBIGUOUS").alias("issue_code"),
    F.lit("leads").alias("source"),
    F.lit("data_nascimento").alias("field_name"),
    F.col("__data_nascimento_raw").cast("string").alias("raw_value"),
    F.lit(None).cast("string").alias("normalized_value"),
)

birth_invalid_df = joined_df.filter(
    F.col("lead_birthdate_status").isin("UNPARSEABLE", "BEFORE_MIN", "FUTURE")
).select(
    F.col("__lead_row_id"),
    F.lit("error").alias("severity"),
    F.lit("LEAD_BIRTHDATE_INVALID").alias("issue_code"),
    F.lit("leads").alias("source"),
    F.lit("data_nascimento").alias("field_name"),
    F.col("__data_nascimento_raw").cast("string").alias("raw_value"),
    F.col("lead_birthdate_candidate").cast("string").alias("normalized_value"),
)

event_ambiguous_df = joined_df.filter(F.col("lead_event_date_status") == F.lit("AMBIGUOUS")).select(
    F.col("__lead_row_id"),
    F.lit("error").alias("severity"),
    F.lit("DATE_AMBIGUOUS").alias("issue_code"),
    F.lit("leads").alias("source"),
    F.lit("data_evento").alias("field_name"),
    F.col("__data_evento_raw").cast("string").alias("raw_value"),
    F.lit(None).cast("string").alias("normalized_value"),
)

event_invalid_df = joined_df.filter(
    F.col("lead_event_date_status").isin("MISSING", "UNPARSEABLE", "BEFORE_MIN")
).select(
    F.col("__lead_row_id"),
    F.lit("error").alias("severity"),
    F.lit("LEAD_EVENT_DATE_INVALID").alias("issue_code"),
    F.lit("leads").alias("source"),
    F.lit("data_evento").alias("field_name"),
    F.col("__data_evento_raw").cast("string").alias("raw_value"),
    F.col("lead_event_date_candidate").cast("string").alias("normalized_value"),
)

context_missing_df = joined_df.filter(F.col("required_context_missing")).select(
    F.col("__lead_row_id"),
    F.lit("error").alias("severity"),
    F.lit("MISSING_REQUIRED_CONTEXT").alias("issue_code"),
    F.lit("leads").alias("source"),
    F.lit("evento|tipo_evento|local").alias("field_name"),
    F.concat_ws(
        " | ",
        F.coalesce(F.col("evento"), F.lit("")),
        F.coalesce(F.col("tipo_evento"), F.lit("")),
        F.coalesce(F.col("local"), F.lit("")),
    ).alias("raw_value"),
    F.lit(None).cast("string").alias("normalized_value"),
)

for frame in [
    cpf_issues_df,
    birth_ambiguous_df,
    birth_invalid_df,
    event_ambiguous_df,
    event_invalid_df,
    context_missing_df,
]:
    issues_df = issues_df.unionByName(frame.select(*ISSUES_COLUMNS), allowMissingColumns=False)

issues_df.createOrReplaceTempView("issues_df")
print("✅ issues_df montado.")

In [ ]:
# CÉLULA 14 — Auditoria e exibição final
def clean_column_name(name):
    n = unicodedata.normalize('NFKD', name).encode('ASCII', 'ignore').decode('ASCII')
    n = re.sub(r'[^a-zA-Z0-9_]', '_', n)
    return re.sub(r'_+', '_', n).strip('_').lower()

new_column_names = [clean_column_name(c) for c in final_df.columns]
final_df_clean = final_df.toDF(*new_column_names)

tabela_producao = "transientes_pd.d5f1a1f.leads"

try:
    print(f"🚀 Iniciando gravação no Catálogo (Modo Serverless)...")
    final_df_clean.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(tabela_producao)
    print(f"✅ SUCESSO ABSOLUTO! Tabela '{tabela_producao}' atualizada.")
    print(f"📊 Total de registros: {final_df_clean.count()}")
    print(f"📂 Colunas: {final_df_clean.columns}")
except Exception as e:
    print(f"❌ Erro ao salvar: {e}")